# 02 · Quantum Gates and Circuits

In notebook 01 we met the qubit, superposition, and entanglement. Now we look at the **operations** that manipulate qubits — **quantum gates** — and how to compose them into **circuits**.

We cover:

1. What a quantum gate is (a reversible, unitary operation)
2. Single-qubit gates: `X`, `Y`, `Z`, `H`, `S`, `T`, and rotations `RX/RY/RZ`
3. Multi-qubit gates: `CNOT`, `CZ`, `SWAP`, `Toffoli`
4. Reading a qubit's exact state with the **`Statevector`** (no sampling noise)
5. Building and running a small multi-gate circuit

Again, everything runs locally with **Qiskit 1.x**.

In [ ]:
# Run once if you haven't already.
%pip install qiskit qiskit-aer matplotlib pylatexenc

## 1. What is a quantum gate?

A quantum gate is a **unitary** operation: it transforms the qubit state in a way that is reversible and preserves total probability (the amplitudes still square-sum to 1). Mathematically each gate is a unitary matrix that multiplies the state vector.

Unlike many classical logic gates (e.g. `AND`, which throws information away), **every quantum gate is reversible** — you can always undo it by applying its inverse.

## 2. Single-qubit gates

| Gate | Effect | Intuition |
|---|---|---|
| `X` | $|0\rangle\leftrightarrow|1\rangle$ | Quantum **NOT** (bit flip) |
| `Y` | bit + phase flip | Rotation about the Y axis by $\pi$ |
| `Z` | $|1\rangle \to -|1\rangle$ | **Phase flip** |
| `H` | basis $\leftrightarrow$ superposition | Creates equal superposition |
| `S` | quarter phase | $\sqrt{Z}$ |
| `T` | eighth phase | $\sqrt{S}$ |
| `RX(θ)`,`RY(θ)`,`RZ(θ)` | rotate by angle θ | **Tunable** gates — the building blocks of quantum ML |

Let's see the bit-flip `X` gate turn $|0\rangle$ into $|1\rangle$.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

qc = QuantumCircuit(1)
qc.x(0)   # flip |0> to |1>

state = Statevector(qc)
print("State after X gate:", state.data)   # expect [0, 1]
qc.draw("mpl")

### Inspecting the exact state

The `Statevector` class computes the *exact* amplitudes (the simulator's privilege — on real hardware you can only measure). Let's confirm that `H` produces equal amplitudes of $1/\sqrt{2} \approx 0.707$.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
print("H|0> amplitudes:", Statevector(qc).data)

# Probabilities are the squared magnitudes of the amplitudes
print("Probabilities  :", Statevector(qc).probabilities_dict())

### Tunable rotation gates

Rotation gates take a continuous **angle** parameter. `RY(θ)` rotates the qubit toward $|1\rangle$, so we can dial in *any* probability — not just 50/50. These parameterized gates are exactly what variational quantum machine-learning models train (see notebook 03).

In [ ]:
import numpy as np

qc = QuantumCircuit(1)
qc.ry(np.pi / 3, 0)   # 60-degree rotation
probs = Statevector(qc).probabilities_dict()
print("P(0), P(1):", probs)   # roughly {'0': 0.75, '1': 0.25}

## 3. Multi-qubit gates

| Gate | Qubits | Effect |
|---|---|---|
| `CNOT` / `cx` | 2 | Flip the **target** if the **control** is $|1\rangle$ |
| `CZ` / `cz` | 2 | Apply a phase flip if both qubits are $|1\rangle$ |
| `SWAP` | 2 | Exchange the states of two qubits |
| `Toffoli` / `ccx` | 3 | Flip the target if **both** controls are $|1\rangle$ |

Controlled gates are how we create **entanglement** and conditional logic. Below, `CNOT` flips qubit 1 because we first set qubit 0 to $|1\rangle$.

In [ ]:
qc = QuantumCircuit(2)
qc.x(0)        # control qubit -> |1>
qc.cx(0, 1)    # therefore target qubit flips to |1> as well
print("State:", Statevector(qc).probabilities_dict())   # expect '11'
qc.draw("mpl")

## 4. Putting it together: a small circuit

Let's build a 3-qubit circuit that mixes single- and multi-qubit gates, then sample it. Reading the diagram left to right is the order operations are applied.

In [ ]:
qc = QuantumCircuit(3)
qc.h(0)            # superposition on qubit 0
qc.cx(0, 1)        # entangle 0 and 1
qc.ry(np.pi / 4, 2)
qc.ccx(0, 1, 2)    # Toffoli: flip qubit 2 if 0 AND 1 are |1>
qc.measure_all()
qc.draw("mpl")

In [ ]:
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram

sampler = StatevectorSampler()
counts = sampler.run([qc], shots=2048).result()[0].data.meas.get_counts()
print(counts)
plot_histogram(counts)

## 5. Gates are reversible

Applying a gate twice in a row often returns the original state (`X·X = I`, `H·H = I`). This reversibility is a defining property of quantum computation.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.h(0)   # undo the first H
print("Back to start:", Statevector(qc).probabilities_dict())   # expect {'0': 1.0}

## Summary

- Quantum gates are **reversible, unitary** operations on qubits.
- **Single-qubit** gates (`X/Y/Z/H/S/T`, rotations) move a single qubit around the Bloch sphere.
- **Multi-qubit** gates (`CNOT/CZ/SWAP/Toffoli`) create entanglement and conditional logic.
- `Statevector` gives the exact amplitudes; `StatevectorSampler` simulates real measurement statistics.
- **Parameterized** rotation gates are the trainable knobs behind quantum machine learning.

**Next:** `03_Quantum_Machine_Learning_Basics.ipynb` — train a variational quantum classifier.

### References
- IBM Quantum Documentation — Qiskit circuits & primitives: https://quantum.cloud.ibm.com/docs/en/guides/simulate-with-qiskit-sdk-primitives
- Qiskit SDK (GitHub): https://github.com/Qiskit/qiskit